In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import networkx as nx
from sklearn.manifold import SpectralEmbedding
import itertools
# import plotly.graph_objects as go
from typing import Set, Dict, Any, List, Tuple
from scipy import sparse
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from typing import Union, Optional
import plotly.express as px


In [3]:
pd.set_option('expand_frame_repr', True)
pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [2]:
def run_dbscan(df, eps=0.15, min_samples=5, scale=False, metric='cosine'):
    """
    Run DBSCAN clustering using vectors stored in 'embedding_json'.

    Args:
        df (pd.DataFrame): Must contain 'embedding_json' as a string like "[0.1, 0.2, ...]".
        eps (float): DBSCAN eps parameter (distance threshold).
        min_samples (int): DBSCAN min_samples parameter.
        scale (bool): Whether to standardize features before clustering.
        metric (str): Distance metric (e.g., 'cosine', 'euclidean').

    Returns:
        X_clustered (ndarray): Embedding matrix for non-noise rows only.
        clustered_df (pd.DataFrame): Rows with cluster labels >= 0 (noise removed).
        noise_df (pd.DataFrame): Rows with cluster == -1.
        summary (dict): {'n_clusters': int, 'n_noise': int}
    """
    df_work = df.copy(deep=True)

    # Parse embeddings
    s = df_work['embedding_json'].astype(str).str.strip()
    s = s.str.lstrip('[').str.rstrip(']')
    s = s.str.split(',')
    vecs = [np.array(list(map(float, parts)), dtype=np.float32) for parts in s]
    X = np.vstack(vecs)

    # Optional scaling
    if scale:
        X = StandardScaler().fit_transform(X)

    # Fit DBSCAN
    db = DBSCAN(eps=eps, min_samples=min_samples, metric=metric)
    labels = db.fit_predict(X)
    df_work['cluster'] = labels

    # Masks
    noise_mask = labels == -1
    cluster_mask = ~noise_mask

    # Split dfs
    noise_df = df_work.loc[noise_mask].drop(columns=['embedding_json']).reset_index(drop=True)
    clustered_df = df_work.loc[cluster_mask].drop(columns=['embedding_json']).reset_index(drop=True)

    # Extract only clustered embeddings
    X_clustered = X[cluster_mask]

    # Summary
    n_clusters = len(set(labels) - {-1})
    n_noise = int(noise_mask.sum())
    summary = {"n_clusters": n_clusters, "n_noise": n_noise}

    return X_clustered, clustered_df, noise_df


In [3]:
d_20250529 = pd.read_csv('data/triples_CO_sbert/goose_20250529.export_triples_sbert.csv')
d_20250530 = pd.read_csv('data/triples_CO_sbert/goose_20250530.export_triples_sbert.csv')
d_20250531 = pd.read_csv('data/triples_CO_sbert/goose_20250531.export_triples_sbert.csv')
d_20250601 = pd.read_csv('data/triples_CO_sbert/goose_20250601.export_triples_sbert.csv')
d_20250602 = pd.read_csv('data/triples_CO_sbert/goose_20250602.export_triples_sbert.csv')
d_20250603 = pd.read_csv('data/triples_CO_sbert/goose_20250603.export_triples_sbert.csv')
d_20250604 = pd.read_csv('data/triples_CO_sbert/goose_20250604.export_triples_sbert.csv')
d_20250605 = pd.read_csv('data/triples_CO_sbert/goose_20250605.export_triples_sbert.csv')
CO_x2d = pd.concat([d_20250529, d_20250530, d_20250531,d_20250601,d_20250602,d_20250603,d_20250604,d_20250605],ignore_index=True)


In [4]:
X,CO_db, c_db = run_dbscan(CO_x2d)

In [5]:
 
fig = px.scatter(
    x=X[:, 0], y=X[:, 1],
    color=CO_db.cluster,
    hover_data={
        "sbert": CO_db['sbert_text'],
        "cluster": CO_db['cluster'],
        "Event" :CO_db['GlobalEventID']


    },
    title="DBSCAN Clusters (first two embedding dimensions)"
)
#fig.update_traces(marker=dict(size=6, opacity=0.7))
fig.show()
